In [1]:
import requests, time, json
import pandas as pd

# ── 발급받은 키 입력 (네이버 개발자센터 → 애플리케이션 → 검색 API) ──
CLIENT_ID     = "Y7n03Br_kFGnH39UTACm"
CLIENT_SECRET = "PNpqoa1XvU"

API_URL = "https://openapi.naver.com/v1/search/shop.json"

def fetch_shopping(query, total=100, sort="sim", exclude="used:rental:cbshop"):
    """query를 total개까지 수집. sort: sim(유사도)/date/asc/dsc"""
    headers = {"X-Naver-Client-Id": CLIENT_ID,
               "X-Naver-Client-Secret": CLIENT_SECRET}
    items, total = [], min(total, 1000)          # 네이버 전체 상한 1000
    for start in range(1, total + 1, 100):
        params = {"query": query,
                  "display": min(100, total - start + 1),   # 요청당 상한 100
                  "start": start, "sort": sort, "exclude": exclude}
        r = requests.get(API_URL, headers=headers, params=params)
        if r.status_code != 200:
            print(f"[{r.status_code}] {r.text}"); break
        batch = r.json().get("items", [])
        if not batch: break
        items.extend(batch)
        time.sleep(0.1)
    return items

In [2]:
# ── 호출 ──
query = "무선이어폰"
items = fetch_shopping(query, total=1000, sort="date")
print(f"수집: {len(items)}건\n")

# ── 1) 원시 JSON 1건 그대로 보기 (더러움 확인) ──
print("─── raw JSON 샘플 ───")
print(json.dumps(items[0], ensure_ascii=False, indent=2))

# ── 2) 표로 보기 (정제 X, 온 그대로) ──
df = pd.DataFrame(items)
cols = ["title","brand","maker","mallName","lprice","hprice",
        "productType","category1","category2","category3","category4"]
df = df[[c for c in cols if c in df.columns]]
display(df.head(10))

df.to_csv(f"naver_{query}.csv", index=False, encoding="utf-8-sig")  # 필요시 저장

수집: 1000건

─── raw JSON 샘플 ───
{
  "title": "PC 노트북 USB 네트워크 <b>무선</b> 블루투스 동글이 윈도우11 지원 동글",
  "link": "https://link.coupang.com/re/PCSNAVERPCSDP?pageKey=9115692765&ctag=9115692765&lptag=V87041344295&itemId=26810746955&vendorItemId=87041344295&spec=10305199",
  "image": "https://shopping-phinf.pstatic.net/main_6067164/60671643681.jpg",
  "lprice": "17000",
  "hprice": "",
  "mallName": "쿠팡",
  "productId": "60671643681",
  "productType": "2",
  "brand": "아이피타임",
  "maker": "",
  "category1": "디지털/가전",
  "category2": "음향가전",
  "category3": "이어폰/헤드폰액세서리",
  "category4": "기타이어폰/헤드폰액세서리"
}


,title,brand,maker,mallName,lprice,hprice,productType,category1,category2,category3,category4
0,PC 노트북 USB 네트워크 <b>무선</b> 블루투스 동글이 윈도우11 지원 동글,아이피타임,,쿠팡,17000,,2,디지털/가전,음향가전,이어폰/헤드폰액세서리,기타이어폰/헤드폰액세서리
1,NAVEE NV2O-WE1 오픈형 블루투스이어폰 귀안아픈 <b>무선이어폰</b> 등...,NAVEE,,쿠팡,30400,,2,디지털/가전,음향가전,이어폰,
2,BEV7 블루투스 <b>이어폰</b> <b>무선</b> 헤드셋 이어셋 보조배터리 ...,마하,,쿠팡,65990,,2,디지털/가전,음향가전,이어폰,
3,통화 음악감상 휴대 <b>무선</b> 블루투스 <b>이어폰</b> 블랙,,,쿠팡,35290,,2,디지털/가전,음향가전,이어폰,
4,아이리버 <b>무선이어폰</b> 오픈핏 귀걸이형 블루투스 IB-iX7 블랙,아이리버,아이리버,쿠팡,50970,,2,디지털/가전,음향가전,이어폰,
5,엠지텍 통화용 2대 멀티포인트 퀄컴 APT-X코덱 <b>무선</b> 블루투스 <b>...,엠지텍,,쿠팡,164200,,2,디지털/가전,음향가전,이어폰,
6,Briwa 블루투스 <b>무선 이어폰</b> 고음질 자동 페어링 블루투스 6.0 투...,,,쿠팡,56800,,2,디지털/가전,음향가전,이어폰,
7,LG 정품 <b>무선</b> 스피커 USB C 타입 1M 케이블 COV3766720...,LG전자,LG전자,쿠팡,19800,,2,디지털/가전,음향가전,이어폰/헤드폰액세서리,연장선/케이블
8,아이리버 ITW-J6 TWS <b>무선</b> 블루투스 5.1 <b>이어폰</b> ...,아이리버,,쿠팡,73840,,2,디지털/가전,음향가전,이어폰,
9,미니트 오픈형 LED디스플레이 5.3 <b>무선</b> 블루투스 <b>이어폰</b>...,,,쿠팡,34500,,2,디지털/가전,음향가전,이어폰,


In [ ]:
# 저장된 CSV 로드 (API 재호출 없이 분석)
import re
import pandas as pd

df = pd.read_csv("naver_무선이어폰_1000.csv", encoding="utf-8-sig")
df["category4"] = df["category4"].fillna("")
print(f"전체 행: {len(df)}개 / 컬럼: {list(df.columns)}")
df.head(5)

전체 행: 1000개 / 컬럼: ['title', 'brand', 'maker', 'mallName', 'lprice', 'hprice', 'productType', 'category1', 'category2', 'category3', 'category4']


,title,brand,maker,mallName,lprice,hprice,productType,category1,category2,category3,category4
0,아웨이 A610BL 블루투스이어폰 <b>무선이어폰</b> 볼륨조절,NaN,NaN,11번가,25820,NaN,2,디지털/가전,음향가전,블루투스셋,블루투스이어폰/이어셋
1,플랜트로닉스 Poly Voyager 5200 <b>무선</b> 헤드셋 소음 제거 마...,POLY,NaN,11번가,228000,NaN,2,디지털/가전,음향가전,블루투스셋,블루투스이어폰/이어셋
2,버브버즈250 블루투스 <b>이어폰</b> vervebuds250,모토로라,NaN,11번가,73900,NaN,2,디지털/가전,음향가전,블루투스셋,블루투스이어폰/이어셋
3,캔디 BE-22 골전도 음성 증폭기 블루투스 <b>무선 이어폰</b> 청력 보호 헤드셋,NaN,NaN,11번가,130000,NaN,2,디지털/가전,음향가전,블루투스셋,블루투스이어폰/이어셋
4,"삼성 갤럭시 버즈 3 프로 AI 트루 <b>와이어리스</b> 블루투스 이어버드, 노...",NaN,NaN,11번가,265950,NaN,2,디지털/가전,음향가전,블루투스셋,블루투스이어폰/이어셋


---
## 데이터 탐색 (EDA)

수집한 1000개 데이터를 살펴보며 두 가지 문제를 발견합니다.

1. **category4 오염** — "무선이어폰"을 검색했는데 이어폰이 아닌 제품이 섞여 있음  
2. **title 노이즈** — 제품명에 쇼핑몰 홍보 문구, HTML 태그 등이 섞여 있음

### 문제 1. category4 — 검색어와 무관한 제품이 섞여 있다

In [ ]:
# category4 분포 확인
cat4_counts = df["category4"].value_counts()
print(cat4_counts.to_string())
print(f"\n고유 카테고리 수: {df['category4'].nunique()}개")
print(f"category4 미분류(빈값): {(df['category4'] == '').sum()}개")

category4
블루투스이어폰/이어셋       575
                  116
생활용무전기             88
블루투스헤드폰/헤드셋        46
오토바이블루투스           37
기타이어폰/헤드폰액세서리      24
충전기                18
거치대                18
케이스/파우치            15
캡/솜/팁              12
음향기기                7
기타 MP3/PMP액세서리      7
기타 전기용품             6
기타유아동퍼즐             3
연장선/케이블             3
크래들                 3
진공청소기               3
2.1채널               2
파워믹서                2
노트                  1
기타수영용품              1
헬멧                  1
타악기소품/부품            1
차량용휴대폰충전기           1
블루투스스피커             1
퍼스널뷰어               1
리모컨                 1
휴대용선풍기              1
금고                  1
무선마이크               1
유무선전화기              1
액세서리                1
블루투스동글              1
필통                  1

고유 카테고리 수: 34개
category4 미분류(빈값): 116개


In [ ]:
# 이어폰과 무관한 제품 예시 확인
noise_cats = ["생활용무전기", "오토바이블루투스", "충전기", "거치대", "케이스/파우치"]
noise_df = df[df["category4"].isin(noise_cats)][["title", "category4"]]
display(noise_df.sample(10, random_state=42).reset_index(drop=True))

,title,category4
0,Kebidumei Y10 블루투스 오토바이 헬멧 헤드셋 <b>무선</b> MP3 플...,생활용무전기
1,VR 로봇 MH01 오토바이 블루투스 5.0 헬멧 헤드셋 핸즈프리 <b>이어폰</b...,생활용무전기
2,A8 프로 오토바이 헬멧 블루투스 헤드셋 <b>무선</b> V5.3 방수 핸즈프리 ...,오토바이블루투스
3,Y11 오토바이 헬멧 블루투스 5.3 헤드셋 방수 모토 헤드폰 <b>무선</b> 핸...,생활용무전기
4,BT90B 블루투스 5.4 오토바이 헬멧 헤드셋 500M 인터콤 2인용 <b>무선<...,생활용무전기
5,Y10 오토바이 헬멧 블루투스 헤드셋 <b>무선</b> 헤드폰 음성 제어 <b>이어...,생활용무전기
6,BT31 오토바이 헬멧 헤드셋 블루투스 5.3 방수 <b>무선 이어폰</b> 마이크...,생활용무전기
7,에어팟 호환 귀걸이형 <b>무선이어폰</b> 고정후크 M13398,거치대
8,<b>무선</b> 오토바이 헬멧 헤드셋 핸즈프리 통화 휴대폰 방수 <b>이어폰</b...,생활용무전기
9,벨킨 울트라차지 2in1 Qi2 25W 폴더블 마그네틱 <b>무선</b> 초고속 충...,충전기


In [ ]:
# 비율로 요약
earphone_cnt = (df["category4"] == "블루투스이어폰/이어셋").sum()
unlabeled_cnt = (df["category4"] == "").sum()
noise_cnt = len(df) - earphone_cnt - unlabeled_cnt

print(f"이어폰 (블루투스이어폰/이어셋) : {earphone_cnt}개 ({earphone_cnt/len(df)*100:.1f}%)")
print(f"관련 없는 제품 (노이즈)        : {noise_cnt}개 ({noise_cnt/len(df)*100:.1f}%)")
print(f"미분류 (빈값)                  : {unlabeled_cnt}개 ({unlabeled_cnt/len(df)*100:.1f}%)")
print("\n→ 실습 1: title만 보고 이어폰/비이어폰을 분류하는 모델 학습")

이어폰 (블루투스이어폰/이어셋) : 575개 (57.5%)
관련 없는 제품 (노이즈)        : 309개 (30.9%)
미분류 (빈값)                  : 116개 (11.6%)

→ 실습 1: title만 보고 이어폰/비이어폰을 분류하는 모델 학습


### 문제 2. title — 제품명에 홍보 문구와 HTML 태그가 섞여 있다

In [ ]:
# title 샘플 출력 — 원본 그대로
print("=== title 원본 샘플 ===")
for t in df["title"].head(10).tolist():
    print(" ", t)

=== title 원본 샘플 ===
  아웨이 A610BL 블루투스이어폰 <b>무선이어폰</b> 볼륨조절
  플랜트로닉스 Poly Voyager 5200 <b>무선</b> 헤드셋 소음 제거 마이크 블루투스
  버브버즈250 블루투스 <b>이어폰</b> vervebuds250
  캔디 BE-22 골전도 음성 증폭기 블루투스 <b>무선 이어폰</b> 청력 보호 헤드셋
  삼성 갤럭시 버즈 3 프로 AI 트루 <b>와이어리스</b> 블루투스 이어버드, 노이즈 캔슬링, 사운드 최적화, 실시간 인터프리터, 새롭게 디자인된 컴포트 핏 라틴 아메리카 버전(실버)
  R500 피닉스 스테레오 <b>이어폰</b> <b>무선</b> 블루투스 SBT
  스마텍 STBT-TW300  블루투스 <b>이어폰</b>
  LG전자 TONE Free 크래들 (HBS-TFN5, 글로시 화이트) COA01245406 (<b>이어폰</b> 미포함)
  [브이와이][VCHILP47]스마텍 부품 블루투스 이어셋 STBT-N2
  브리츠 TWS 블루투스 <b>이어폰</b> C타입 충전 <b>무선</b> 이어셋
